# Wadjet v2 — Hieroglyph Detector v2 (YOLO26s Resume Training)

**Resume from v1 checkpoint** (epoch 88/150 — timed out on Kaggle 12h limit)

| Property | Value |
|----------|-------|
| Architecture | YOLO26s (NMS-free end-to-end) |
| Input | `[1, 3, 640, 640]` float32, /255 normalized |
| Output | `[1, 300, 6]` = (batch, max_dets, [x1,y1,x2,y2,conf,cls]) |
| Classes | 1 ("hieroglyph") |
| Training data | 10,311 images (5 sources, YOLO format) |
| Resume from | v1 `last.pt` at epoch 88, mAP50=0.710 |
| Checkpoint source | `kernel_sources: nadermohamedcr7/wadjet-hieroglyph-detector` |

**Strategy**: Fine-tune from v1 `last.pt` for 60 more epochs with lower LR (fits 12h limit).

**CRITICAL RULES:**
- **NO horizontal flip** (`fliplr=0.0`) — hieroglyph orientation = meaning
- **NO vertical flip** (`flipud=0.0`)
- float32 precision only (no AMP/mixed precision)
- End-to-end ONNX export (NMS-free, output `[1, 300, 6]`)

In [ ]:
# Cell 1: Install dependencies & KeepAlive setup
!pip install -U ultralytics --quiet
!pip install -q onnxscript onnxruntime

import os, sys, json, time, shutil, zipfile, threading
from pathlib import Path

import torch
import onnx
import onnxruntime as ort
import numpy as np

import ultralytics
from ultralytics import YOLO, settings

print(f"Python:       {sys.version}")
print(f"PyTorch:      {torch.__version__}")
print(f"CUDA:         {torch.cuda.is_available()} ({torch.version.cuda})")
if torch.cuda.is_available():
    print(f"GPU:          {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM:         {vram:.1f} GB")
print(f"ONNX Runtime: {ort.__version__}")
print(f"Ultralytics:  {ultralytics.__version__}")

# Verify YOLO26 is available
try:
    _test = YOLO("yolo26s.pt")
    del _test
    print("YOLO26s:      AVAILABLE")
except Exception as e:
    print(f"YOLO26s:      NOT AVAILABLE — {e}")
    print("Try: pip install -U ultralytics")

# KeepAlive — prevents IOPub timeout (papermill kills cells with >4s silence)
stop_keepalive = threading.Event()

def start_keepalive():
    """Start a background thread that prints a dot every 30s."""
    stop_keepalive.clear()
    def _run():
        while not stop_keepalive.is_set():
            stop_keepalive.wait(30)
            if not stop_keepalive.is_set():
                print(".", end="", flush=True)
    t = threading.Thread(target=_run, daemon=True)
    t.start()
    return t

def stop_keepalive_fn():
    stop_keepalive.set()

print("\nKeepAlive ready.", flush=True)

In [ ]:
# Cell 2: Find v1 checkpoint (last.pt uploaded as dataset)
# The checkpoint is uploaded as dataset "wadjet-detector-checkpoint" containing last.pt

CHECKPOINT = None
CKPT_DATASET_SLUG = "wadjet-detector-checkpoint"

print("=== Searching for v1 checkpoint ===", flush=True)

# Search strategy: datasets appear under /kaggle/input/<dataset-slug>/
search_paths = [
    f"/kaggle/input/{CKPT_DATASET_SLUG}",
    "/kaggle/input",
]

# Deep listing for debug
for base in search_paths:
    if not os.path.isdir(base):
        print(f"  {base}/ -- not found", flush=True)
        continue
    print(f"\n{base}/", flush=True)
    for root, dirs, files in os.walk(base):
        depth = root.replace(base, "").count(os.sep)
        if depth > 4:
            dirs.clear()
            continue
        indent = "  " * depth
        print(f"{indent}{os.path.basename(root)}/", flush=True)
        for f in files[:10]:
            fpath = os.path.join(root, f)
            sz = os.path.getsize(fpath) / 1024**2
            print(f"{indent}  {f} ({sz:.1f} MB)", flush=True)
        if len(files) > 10:
            print(f"{indent}  ... +{len(files)-10} more", flush=True)

# Find last.pt or best.pt (prefer last.pt -- has optimizer state for resume)
for base in search_paths:
    if not os.path.isdir(base):
        continue
    for root, dirs, files in os.walk(base):
        for fname in ["last.pt", "best.pt"]:
            if fname in files:
                candidate = os.path.join(root, fname)
                sz = os.path.getsize(candidate) / 1024**2
                print(f"\nFound: {candidate} ({sz:.1f} MB)", flush=True)
                if CHECKPOINT is None:
                    CHECKPOINT = candidate

if CHECKPOINT is None:
    print("\nWARNING: No checkpoint found! Will train from scratch with yolo26s.pt")
    CHECKPOINT = "yolo26s.pt"
    FROM_SCRATCH = True
else:
    FROM_SCRATCH = False
    print(f"\nUsing checkpoint: {CHECKPOINT}")

print(f"Resume from scratch: {FROM_SCRATCH}", flush=True)

In [ ]:
# Cell 3: Auto-discover dataset root (3-level search)
DATA_ROOT = None
DATASET_SLUG = "wadjet-hieroglyph-detection"

print("=== Auto-discovering dataset ===", flush=True)

# Strategy: search /kaggle/input for images/ + labels/ with train/val/test
for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.replace("/kaggle/input", "").count(os.sep)
    if depth > 4:
        dirs.clear()
        continue
    if "images" in dirs and "labels" in dirs:
        img_train = os.path.join(root, "images", "train")
        if os.path.isdir(img_train):
            DATA_ROOT = root
            break

# If not found, try extracting zips
if DATA_ROOT is None:
    print("No direct dataset. Searching zips...", flush=True)
    extract_dir = "/kaggle/working/dataset"
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f.endswith(".zip"):
                zpath = os.path.join(root, f)
                print(f"Extracting {zpath}...", flush=True)
                os.makedirs(extract_dir, exist_ok=True)
                with zipfile.ZipFile(zpath, "r") as zf:
                    zf.extractall(extract_dir)
                for r2, d2, f2 in os.walk(extract_dir):
                    if "images" in d2 and "labels" in d2:
                        if os.path.isdir(os.path.join(r2, "images", "train")):
                            DATA_ROOT = r2
                            break
                if DATA_ROOT:
                    break
        if DATA_ROOT:
            break

if DATA_ROOT is None:
    raise FileNotFoundError("Dataset not found under /kaggle/input/!")

# Verify all required dirs
for split in ["train", "val", "test"]:
    for sub in ["images", "labels"]:
        d = os.path.join(DATA_ROOT, sub, split)
        assert os.path.isdir(d), f"Missing: {d}"

print(f"DATA_ROOT: {DATA_ROOT}")
print(f"Contents:  {sorted(os.listdir(DATA_ROOT))}")
for split in ["train", "val", "test"]:
    n_img = len(os.listdir(os.path.join(DATA_ROOT, "images", split)))
    n_lbl = len(os.listdir(os.path.join(DATA_ROOT, "labels", split)))
    print(f"  {split:5s}: {n_img:,} images, {n_lbl:,} labels")
print(flush=True)

In [ ]:
# Cell 4: Dataset validation & statistics
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

print("=== Dataset Validation ===", flush=True)

total_boxes = 0
empty_labels = 0
bad_labels = 0

for split in ["train", "val", "test"]:
    lbl_dir = os.path.join(DATA_ROOT, "labels", split)
    img_dir = os.path.join(DATA_ROOT, "images", split)
    n_img = len(os.listdir(img_dir))
    n_lbl = len(os.listdir(lbl_dir))
    split_boxes = 0
    split_empty = 0

    for lf in sorted(os.listdir(lbl_dir))[:500]:  # Check first 500
        fpath = os.path.join(lbl_dir, lf)
        with open(fpath) as fh:
            lines = [l.strip() for l in fh if l.strip()]
        if not lines:
            split_empty += 1
            continue
        for line in lines:
            parts = line.split()
            if len(parts) != 5 or parts[0] != "0":
                bad_labels += 1
            split_boxes += 1

    total_boxes += split_boxes
    empty_labels += split_empty
    avg = split_boxes / min(n_lbl, 500) if min(n_lbl, 500) > 0 else 0
    print(f"  {split:5s}: {n_img:,} imgs, {n_lbl:,} lbls, ~{avg:.1f} boxes/img (sampled 500)")

print(f"\n  Total boxes (sampled): {total_boxes:,}")
print(f"  Empty labels: {empty_labels}")
print(f"  Bad labels: {bad_labels}")

# Visualize 6 random train images with boxes
img_dir = os.path.join(DATA_ROOT, "images", "train")
lbl_dir = os.path.join(DATA_ROOT, "labels", "train")
all_imgs = sorted(os.listdir(img_dir))
rng = np.random.default_rng(42)
sample_idxs = rng.choice(len(all_imgs), size=min(6, len(all_imgs)), replace=False)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, idx in zip(axes.flat, sample_idxs):
    img_name = all_imgs[idx]
    img_path = os.path.join(img_dir, img_name)
    lbl_path = os.path.join(lbl_dir, Path(img_name).stem + ".txt")

    img = Image.open(img_path)
    w, h = img.size
    ax.imshow(img)
    ax.set_title(img_name[:30], fontsize=8)
    ax.axis("off")

    if os.path.isfile(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                    x1 = (cx - bw/2) * w
                    y1 = (cy - bh/2) * h
                    rect = patches.Rectangle((x1, y1), bw*w, bh*h,
                                             linewidth=1, edgecolor='lime', facecolor='none')
                    ax.add_patch(rect)

plt.tight_layout()
plt.savefig("/kaggle/working/sample_annotations.png", dpi=100)
plt.show()
print("Validation complete.", flush=True)

In [ ]:
# Cell 5: Configure & Train YOLO26s (resume from checkpoint)
import yaml

# ===================== TRAINING CONFIG =====================
IMG_SIZE     = 640
EPOCHS       = 60           # Remaining epochs (fits 12h: 60 * 8min = 8h)
PATIENCE     = 25           # Slightly less than v1's 30
BATCH_SIZE   = 16
WARMUP_EP    = 3            # Short warmup since already trained
LR0          = 0.002        # Lower LR for fine-tuning (v1 used 0.01)
LRF          = 0.01

# Augmentation — NO FLIPS for hieroglyphs
FLIPLR       = 0.0
FLIPUD       = 0.0
MOSAIC       = 1.0
MIXUP        = 0.1
CLOSE_MOSAIC = 10           # Disable mosaic at epoch 50/60
DEGREES      = 15.0
TRANSLATE    = 0.1
SCALE        = 0.5
PERSPECTIVE  = 0.001
HSV_H        = 0.015
HSV_S        = 0.7
HSV_V        = 0.4

# Quality gates
MIN_MAP50      = 0.75       # Realistic target given v1 plateau
MIN_PRECISION  = 0.75
MIN_RECALL     = 0.68
MAX_ONNX_MB    = 20
EXPECTED_SHAPE = (1, 300, 6)

OUTPUT_DIR   = "/kaggle/working"
PROJECT_DIR  = os.path.join(OUTPUT_DIR, "detector_v2")

# Create data.yaml with absolute paths
ds_config = {
    "path": DATA_ROOT,
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 1,
    "names": ["hieroglyph"],
}
FIXED_YAML = os.path.join(OUTPUT_DIR, "hieroglyph_det.yaml")
with open(FIXED_YAML, "w") as f:
    yaml.dump(ds_config, f, default_flow_style=False)

print("Config:")
print(f"  Checkpoint: {CHECKPOINT}")
print(f"  From scratch: {FROM_SCRATCH}")
print(f"  Epochs: {EPOCHS}, LR0: {LR0}, Batch: {BATCH_SIZE}")
print(f"  fliplr={FLIPLR}, flipud={FLIPUD}")
print(f"  Quality gates: mAP50>={MIN_MAP50}, P>={MIN_PRECISION}, R>={MIN_RECALL}")
print(f"  YAML: {FIXED_YAML}")
print(flush=True)

# Configure ultralytics
settings.update({"runs_dir": PROJECT_DIR, "sync": False, "wandb": False})
os.environ["YOLO_VERBOSE"] = "True"

# Start KeepAlive
ka = start_keepalive()

print("=" * 60, flush=True)
print("STARTING YOLO26s TRAINING (RESUME)" if not FROM_SCRATCH else "STARTING YOLO26s TRAINING (FRESH)", flush=True)
print("=" * 60, flush=True)

t0 = time.time()

# Load checkpoint
model = YOLO(CHECKPOINT)

# Train
results = model.train(
    data=FIXED_YAML,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    patience=PATIENCE,
    optimizer="auto",
    lr0=LR0,
    lrf=LRF,
    warmup_epochs=WARMUP_EP,
    # Augmentation — NO FLIPS
    fliplr=FLIPLR,
    flipud=FLIPUD,
    mosaic=MOSAIC,
    mixup=MIXUP,
    close_mosaic=CLOSE_MOSAIC,
    degrees=DEGREES,
    translate=TRANSLATE,
    scale=SCALE,
    perspective=PERSPECTIVE,
    hsv_h=HSV_H,
    hsv_s=HSV_S,
    hsv_v=HSV_V,
    # Training settings
    amp=False,               # NO mixed precision — clean ONNX
    workers=2,
    project=PROJECT_DIR,
    name="yolo26s_hiero_v2",
    exist_ok=True,
    verbose=True,
    save=True,
    save_period=20,
    plots=True,
)

stop_keepalive_fn()
elapsed = (time.time() - t0) / 60
print(f"\nTraining complete in {elapsed:.1f} min", flush=True)

In [ ]:
# Cell 6: Training metrics & quality gate checks
import csv

# Find best.pt
train_dir = os.path.join(PROJECT_DIR, "yolo26s_hiero_v2")
best_pt = os.path.join(train_dir, "weights", "best.pt")

if not os.path.isfile(best_pt):
    for root, dirs, files in os.walk(PROJECT_DIR):
        if "best.pt" in files:
            best_pt = os.path.join(root, "best.pt")
            break

if not os.path.isfile(best_pt):
    raise FileNotFoundError(f"best.pt not found under {PROJECT_DIR}")

print(f"Best model: {best_pt}")
print(f"Size: {os.path.getsize(best_pt) / 1024**2:.1f} MB", flush=True)

# Load and plot results CSV
results_csv = os.path.join(train_dir, "results.csv")
if os.path.isfile(results_csv):
    with open(results_csv) as f:
        reader = csv.DictReader(f)
        rows = [r for r in reader]

    epochs_list = [int(r["epoch"].strip()) for r in rows]
    box_loss = [float(r["train/box_loss"].strip()) for r in rows]
    cls_loss = [float(r["train/cls_loss"].strip()) for r in rows]
    val_box  = [float(r["val/box_loss"].strip()) for r in rows]
    map50s   = [float(r["metrics/mAP50(B)"].strip()) for r in rows]
    map95s   = [float(r["metrics/mAP50-95(B)"].strip()) for r in rows]
    precs    = [float(r["metrics/precision(B)"].strip()) for r in rows]
    recs     = [float(r["metrics/recall(B)"].strip()) for r in rows]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0,0].plot(epochs_list, box_loss, label="train/box_loss")
    axes[0,0].plot(epochs_list, val_box, label="val/box_loss")
    axes[0,0].set_title("Box Loss"); axes[0,0].legend(); axes[0,0].grid(True)

    axes[0,1].plot(epochs_list, cls_loss, label="train/cls_loss")
    axes[0,1].set_title("Cls Loss"); axes[0,1].legend(); axes[0,1].grid(True)

    axes[1,0].plot(epochs_list, map50s, label="mAP50", color="green")
    axes[1,0].plot(epochs_list, map95s, label="mAP50-95", color="blue")
    axes[1,0].axhline(y=MIN_MAP50, color="red", linestyle="--", label=f"gate={MIN_MAP50}")
    axes[1,0].set_title("mAP"); axes[1,0].legend(); axes[1,0].grid(True)

    axes[1,1].plot(epochs_list, precs, label="Precision", color="orange")
    axes[1,1].plot(epochs_list, recs, label="Recall", color="purple")
    axes[1,1].axhline(y=MIN_PRECISION, color="red", linestyle="--", label=f"P gate={MIN_PRECISION}")
    axes[1,1].axhline(y=MIN_RECALL, color="red", linestyle=":", label=f"R gate={MIN_RECALL}")
    axes[1,1].set_title("P/R"); axes[1,1].legend(); axes[1,1].grid(True)

    plt.tight_layout()
    plt.savefig("/kaggle/working/training_curves.png", dpi=100)
    plt.show()

# Validate on val set
best_model = YOLO(best_pt)
val_metrics = best_model.val(data=FIXED_YAML, split="val", verbose=True)

map50 = val_metrics.box.map50
map50_95 = val_metrics.box.map
precision = val_metrics.box.mp
recall = val_metrics.box.mr

print(f"\n{'='*50}")
print(f"VALIDATION RESULTS")
print(f"{'='*50}")
print(f"  mAP50:      {map50:.4f}  (gate: >= {MIN_MAP50})")
print(f"  mAP50-95:   {map50_95:.4f}")
print(f"  Precision:  {precision:.4f}  (gate: >= {MIN_PRECISION})")
print(f"  Recall:     {recall:.4f}  (gate: >= {MIN_RECALL})")

gates_passed = True
for name, val, gate in [("mAP50", map50, MIN_MAP50), ("Precision", precision, MIN_PRECISION), ("Recall", recall, MIN_RECALL)]:
    status = "PASS" if val >= gate else "FAIL"
    if val < gate:
        gates_passed = False
    print(f"  [{status}] {name}: {val:.4f} >= {gate}")

if gates_passed:
    print("\n  ALL QUALITY GATES PASSED")
else:
    print("\n  WARNING: Some quality gates failed. Will still export — review before deploying.")
print(flush=True)

In [ ]:
# Cell 7: Evaluate on test set
print("Evaluating on TEST set...", flush=True)
test_metrics = best_model.val(data=FIXED_YAML, split="test", verbose=True)

test_map50 = test_metrics.box.map50
test_map50_95 = test_metrics.box.map
test_precision = test_metrics.box.mp
test_recall = test_metrics.box.mr

print(f"\n{'='*50}")
print(f"TEST RESULTS")
print(f"{'='*50}")
print(f"  mAP50:      {test_map50:.4f}")
print(f"  mAP50-95:   {test_map50_95:.4f}")
print(f"  Precision:  {test_precision:.4f}")
print(f"  Recall:     {test_recall:.4f}")
print(flush=True)

In [ ]:
# Cell 8: Export to ONNX (fp32, end-to-end NMS-free)
print("Exporting to ONNX (end-to-end, NMS-free)...", flush=True)

onnx_path = best_model.export(
    format="onnx",
    imgsz=IMG_SIZE,
    simplify=True,
    opset=17,
    dynamic=True,      # dynamic batch axis
    half=False,        # keep fp32
)

print(f"ONNX exported: {onnx_path}")
fp32_size = os.path.getsize(onnx_path) / 1024**2
print(f"Size: {fp32_size:.1f} MB")

# Verify no .onnx.data sidecar (dynamo=True creates this)
sidecar = onnx_path + ".data"
if os.path.isfile(sidecar):
    print(f"WARNING: .onnx.data sidecar found ({os.path.getsize(sidecar)/1024**2:.1f} MB)")
    print("This means dynamo was used. Model may not be portable.")
else:
    print("No .onnx.data sidecar — good (single file)")

# Validate ONNX I/O
model_onnx = onnx.load(onnx_path)
print(f"\nONNX graph inputs:")
for inp in model_onnx.graph.input:
    shape = [d.dim_value if d.dim_value else d.dim_param for d in inp.type.tensor_type.shape.dim]
    print(f"  {inp.name}: {shape}")
print(f"ONNX graph outputs:")
for out in model_onnx.graph.output:
    shape = [d.dim_value if d.dim_value else d.dim_param for d in out.type.tensor_type.shape.dim]
    print(f"  {out.name}: {shape}")

# Run dummy inference
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
input_name = sess.get_inputs()[0].name
dummy = np.random.rand(1, 3, IMG_SIZE, IMG_SIZE).astype(np.float32)
outputs = sess.run(None, {input_name: dummy})

print(f"\nDummy inference output shape: {outputs[0].shape}")
print(f"Output dtype: {outputs[0].dtype}")

out_shape = outputs[0].shape
if len(out_shape) == 3:
    print(f"Max detections: {out_shape[1]}, Columns: {out_shape[2]}")
    if out_shape[2] >= 5:
        print("Format: [batch, max_dets, (x1,y1,x2,y2,conf,...)]")
        print("ONNX fp32 export PASSED")
else:
    print(f"WARNING: Unexpected output shape {out_shape}")
print(flush=True)

In [ ]:
# Cell 9: Quantize ONNX to uint8
from onnxruntime.quantization import quantize_dynamic, QuantType

onnx_uint8 = os.path.join(OUTPUT_DIR, "glyph_detector_uint8.onnx")

print("Quantizing to uint8...", flush=True)
quantize_dynamic(
    onnx_path,
    onnx_uint8,
    weight_type=QuantType.QUInt8,
)

uint8_size = os.path.getsize(onnx_uint8) / 1024**2
print(f"  fp32:  {fp32_size:.1f} MB")
print(f"  uint8: {uint8_size:.1f} MB")
print(f"  Compression: {(1 - uint8_size/fp32_size)*100:.0f}%")

# Validate uint8 model
sess_q = ort.InferenceSession(onnx_uint8, providers=["CPUExecutionProvider"])
input_name_q = sess_q.get_inputs()[0].name
outputs_q = sess_q.run(None, {input_name_q: dummy})
print(f"  uint8 output shape: {outputs_q[0].shape}")

# Size gate
if uint8_size > MAX_ONNX_MB:
    print(f"  FAIL: uint8 model {uint8_size:.1f} MB > {MAX_ONNX_MB} MB")
else:
    print(f"  PASS: Size gate {uint8_size:.1f} MB <= {MAX_ONNX_MB} MB")

print(flush=True)

In [ ]:
# Cell 10: Validate ONNX output on a real test image
from PIL import Image

def letterbox_preprocess(img_path, target_size=640):
    """Preprocess image with letterbox padding (same as YOLO)."""
    img = Image.open(img_path).convert("RGB")
    orig_w, orig_h = img.size

    ratio = min(target_size / orig_w, target_size / orig_h)
    new_w = int(orig_w * ratio)
    new_h = int(orig_h * ratio)
    img_resized = img.resize((new_w, new_h), Image.BILINEAR)

    # Pad to target_size
    canvas = Image.new("RGB", (target_size, target_size), (114, 114, 114))
    pad_x = (target_size - new_w) // 2
    pad_y = (target_size - new_h) // 2
    canvas.paste(img_resized, (pad_x, pad_y))

    # To numpy [1, 3, H, W] float32 /255
    arr = np.array(canvas, dtype=np.float32) / 255.0
    arr = arr.transpose(2, 0, 1)[np.newaxis, ...]  # [1, 3, 640, 640]
    return arr, (orig_w, orig_h), ratio, (pad_x, pad_y)

# Pick a real test image
test_img_dir = os.path.join(DATA_ROOT, "images", "test")
test_images = sorted(os.listdir(test_img_dir))
sample_img = os.path.join(test_img_dir, test_images[0])

input_tensor, (ow, oh), ratio, (px, py) = letterbox_preprocess(sample_img, IMG_SIZE)
print(f"Test image: {test_images[0]}")
print(f"Original: {ow}x{oh}, Input shape: {input_tensor.shape}")

# Run uint8 model
outputs = sess_q.run(None, {input_name_q: input_tensor})
preds = outputs[0][0]  # [300, 6]
print(f"Raw output shape: {outputs[0].shape}")
print(f"Predictions per image: {preds.shape[0]}")
print(f"Columns: {preds.shape[1]} ([x1,y1,x2,y2,conf,cls])")

# Filter by confidence
CONF_THRESH = 0.25
mask = preds[:, 4] > CONF_THRESH
filtered = preds[mask]
print(f"\nDetections at conf>{CONF_THRESH}: {len(filtered)}")

if len(filtered) > 0:
    confs = filtered[:, 4]
    print(f"  Confidence: min={confs.min():.3f}, max={confs.max():.3f}, mean={confs.mean():.3f}")
    print(f"  Top 5 confidences: {sorted(confs, reverse=True)[:5]}")

    # Rescale boxes from letterboxed coords to original image coords
    boxes = filtered[:, :4].copy()
    boxes[:, [0, 2]] = (boxes[:, [0, 2]] - px) / ratio
    boxes[:, [1, 3]] = (boxes[:, [1, 3]] - py) / ratio
    widths = boxes[:, 2] - boxes[:, 0]
    heights = boxes[:, 3] - boxes[:, 1]
    print(f"  Box widths:  min={widths.min():.0f}, max={widths.max():.0f}")
    print(f"  Box heights: min={heights.min():.0f}, max={heights.max():.0f}")
else:
    print("  No detections above threshold")

print("\nONNX output format validation PASSED", flush=True)

In [ ]:
# Cell 11: Test on all test set images — detection rate & AI fallback rate
print("=== Running inference on ALL test images ===", flush=True)

test_img_dir = os.path.join(DATA_ROOT, "images", "test")
test_images = sorted(os.listdir(test_img_dir))
total_test = len(test_images)

detected_counts = []
zero_detection_images = []

for i, img_name in enumerate(test_images):
    img_path = os.path.join(test_img_dir, img_name)
    inp, _, _, _ = letterbox_preprocess(img_path, IMG_SIZE)
    out = sess_q.run(None, {input_name_q: inp})
    preds = out[0][0]
    n_det = int((preds[:, 4] > CONF_THRESH).sum())
    detected_counts.append(n_det)
    if n_det == 0:
        zero_detection_images.append(img_name)
    if (i + 1) % 50 == 0:
        print(f"  Processed {i+1}/{total_test}...", flush=True)

detected_counts = np.array(detected_counts)
images_with_dets = int((detected_counts > 0).sum())
images_no_dets = int((detected_counts == 0).sum())
fallback_rate = images_no_dets / total_test

print(f"\n{'='*50}")
print(f"TEST SET DETECTION SUMMARY ({total_test} images)")
print(f"{'='*50}")
print(f"  Images with detections: {images_with_dets}/{total_test} ({images_with_dets/total_test*100:.1f}%)")
print(f"  Images with 0 detections (AI fallback): {images_no_dets}/{total_test} ({fallback_rate*100:.1f}%)")
print(f"  Detection counts: min={detected_counts.min()}, max={detected_counts.max()}, mean={detected_counts.mean():.1f}")
print(f"  AI fallback rate: {fallback_rate*100:.1f}% (gate: < 50%)")

if fallback_rate < 0.50:
    print("  [PASS] AI fallback rate < 50%")
else:
    print("  [FAIL] AI fallback rate >= 50%")

# Visualize 6 test images with detections
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
sample_idxs = rng.choice(len(test_images), size=min(6, len(test_images)), replace=False)

for ax, idx in zip(axes.flat, sample_idxs):
    img_name = test_images[idx]
    img_path = os.path.join(test_img_dir, img_name)
    img = Image.open(img_path).convert("RGB")
    ow, oh = img.size

    inp, _, ratio, (px, py) = letterbox_preprocess(img_path, IMG_SIZE)
    out = sess_q.run(None, {input_name_q: inp})
    preds = out[0][0]
    mask = preds[:, 4] > CONF_THRESH
    dets = preds[mask]

    ax.imshow(img)
    ax.set_title(f"{img_name[:25]} ({len(dets)} dets)", fontsize=8)
    ax.axis("off")

    for det in dets:
        x1 = (det[0] - px) / ratio
        y1 = (det[1] - py) / ratio
        x2 = (det[2] - px) / ratio
        y2 = (det[3] - py) / ratio
        conf = det[4]
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                 linewidth=1.5, edgecolor='lime', facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1-2, f"{conf:.2f}", fontsize=6, color='lime')

plt.tight_layout()
plt.savefig("/kaggle/working/test_detections.png", dpi=100)
plt.show()
print(flush=True)

In [ ]:
# Cell 12: Save artifacts & model metadata
actual_shape = list(outputs_q[0].shape)

metadata = {
    "model_name": "wadjet_hieroglyph_detector_v2",
    "architecture": "YOLO26s (end-to-end, NMS-free)",
    "framework": "ultralytics",
    "task": "object_detection",
    "classes": ["hieroglyph"],
    "num_classes": 1,
    "input_size": IMG_SIZE,
    "input_shape": [1, 3, IMG_SIZE, IMG_SIZE],
    "input_format": "NCHW",
    "normalization": "divide_by_255",
    "input_range": [0.0, 1.0],
    "output_shape": actual_shape,
    "output_columns": ["x1", "y1", "x2", "y2", "confidence", "class_id"],
    "output_format": "[batch, max_detections, 6]",
    "nms_free": True,
    "quantized": True,
    "quantization": "uint8_dynamic",
    "fp32_size_mb": round(fp32_size, 1),
    "uint8_size_mb": round(uint8_size, 1),
    "resumed_from": "v1 epoch 88" if not FROM_SCRATCH else "scratch",
    "training": {
        "dataset": "nadermohamedcr7/wadjet-hieroglyph-detection",
        "train_images": len(os.listdir(os.path.join(DATA_ROOT, "images", "train"))),
        "val_images": len(os.listdir(os.path.join(DATA_ROOT, "images", "val"))),
        "test_images": len(os.listdir(os.path.join(DATA_ROOT, "images", "test"))),
        "v2_epochs": EPOCHS,
        "v1_epochs": 88,
        "total_effective_epochs": 88 + EPOCHS if not FROM_SCRATCH else EPOCHS,
        "batch_size": BATCH_SIZE,
        "img_size": IMG_SIZE,
        "lr0": LR0,
        "fliplr": FLIPLR,
        "flipud": FLIPUD,
        "amp": False,
        "pretrained": CHECKPOINT,
    },
    "metrics": {
        "val_mAP50": round(float(map50), 4),
        "val_mAP50_95": round(float(map50_95), 4),
        "val_precision": round(float(precision), 4),
        "val_recall": round(float(recall), 4),
        "test_mAP50": round(float(test_map50), 4),
        "test_mAP50_95": round(float(test_map50_95), 4),
        "test_precision": round(float(test_precision), 4),
        "test_recall": round(float(test_recall), 4),
        "test_images_with_detections": int(images_with_dets),
        "test_total_images": int(total_test),
        "ai_fallback_rate": round(float(fallback_rate), 4),
    },
    "quality_gates": {
        "mAP50_gate": MIN_MAP50,
        "mAP50_passed": bool(map50 >= MIN_MAP50),
        "precision_gate": MIN_PRECISION,
        "precision_passed": bool(precision >= MIN_PRECISION),
        "recall_gate": MIN_RECALL,
        "recall_passed": bool(recall >= MIN_RECALL),
        "onnx_size_gate_mb": MAX_ONNX_MB,
        "onnx_size_passed": bool(uint8_size <= MAX_ONNX_MB),
        "fallback_rate_gate": 0.50,
        "fallback_rate_passed": bool(fallback_rate < 0.50),
    },
}

meta_path = os.path.join(OUTPUT_DIR, "model_metadata.json")
with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2)
print(f"Metadata saved: {meta_path}")

# Copy artifacts to /kaggle/working
for src, name in [(onnx_uint8, "glyph_detector_uint8.onnx"),
                  (onnx_path, os.path.basename(onnx_path)),
                  (best_pt, "best.pt"),
                  (meta_path, "model_metadata.json")]:
    dst = os.path.join(OUTPUT_DIR, name)
    if src != dst and os.path.isfile(src):
        shutil.copy2(src, dst)

# Copy training plots
for fname in ["results.csv", "results.png", "confusion_matrix.png",
              "confusion_matrix_normalized.png", "P_curve.png",
              "R_curve.png", "PR_curve.png", "F1_curve.png"]:
    src = os.path.join(train_dir, fname)
    if os.path.isfile(src):
        shutil.copy2(src, os.path.join(OUTPUT_DIR, fname))

print("\nOutput files:")
for root, dirs, files in os.walk(OUTPUT_DIR):
    depth = root.replace(OUTPUT_DIR, "").count(os.sep)
    if depth > 2:
        continue
    for f in sorted(files):
        fpath = os.path.join(root, f)
        size = os.path.getsize(fpath)
        rel = os.path.relpath(fpath, OUTPUT_DIR)
        if size > 1024*1024:
            print(f"  {rel:50s} {size/1024**2:8.1f} MB")
        else:
            print(f"  {rel:50s} {size/1024:8.1f} KB")

print(flush=True)

In [ ]:
# Cell 13: Final summary
print("\n" + "=" * 60)
print("TRAINING SUMMARY — YOLO26s Hieroglyph Detector v2")
print("=" * 60)
print(f"  Resumed from:   {'v1 epoch 88' if not FROM_SCRATCH else 'scratch'}")
print(f"  v2 Epochs:      {EPOCHS}")
print(f"  Total effective: {88 + EPOCHS if not FROM_SCRATCH else EPOCHS} epochs")
print(f"  Dataset:        {metadata['training']['train_images']:,} train / "
      f"{metadata['training']['val_images']:,} val / {metadata['training']['test_images']:,} test")
print(f"  ONNX uint8:     {uint8_size:.1f} MB")
print(f"  Output shape:   {actual_shape}")
print()
print(f"  VAL  mAP50={map50:.4f}  P={precision:.4f}  R={recall:.4f}")
print(f"  TEST mAP50={test_map50:.4f}  P={test_precision:.4f}  R={test_recall:.4f}")
print(f"  Fallback rate:  {fallback_rate*100:.1f}%")
print()

all_pass = True
checks = [
    (f"mAP50 >= {MIN_MAP50}",     map50 >= MIN_MAP50),
    (f"Precision >= {MIN_PRECISION}", precision >= MIN_PRECISION),
    (f"Recall >= {MIN_RECALL}",    recall >= MIN_RECALL),
    (f"ONNX <= {MAX_ONNX_MB}MB",  uint8_size <= MAX_ONNX_MB),
    ("Fallback < 50%",            fallback_rate < 0.50),
]

for name, passed in checks:
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_pass = False
    print(f"  [{status}] {name}")

print()
if all_pass:
    print("  ALL GATES PASSED — Model ready for deployment!")
else:
    print("  SOME GATES FAILED — Review before deploying.")

print(f"\nDownload: kaggle kernels output nadermohamedcr7/wadjet-hieroglyph-detector-v2 -p ./detector_output")
print("Done.", flush=True)